In [ ]:
import pandas as pd
import numpy as np

# =====================================================
# 0. 파일 설정
# =====================================================

TOTAL_FILE = "tottenham_youtube_cafe_concat.xlsx"
LABELED_FILE = "tottenham_reanalyze_target_labeled.xlsx"

OUTPUT_FINAL_CSV = "tottenham_youtube_cafe_final_labeled.csv"
OUTPUT_FINAL_XLSX = "tottenham_youtube_cafe_final_labeled.xlsx"

FINAL_COLUMNS = [
    "source",
    "round",
    "club",
    "title",
    "content_type",
    "text_original",
    "text_clean",
    "content_published_at",
    "comment_published_at",
    "like_count",
    "url",
    "period",
    "gpt_sentiment",
    "gpt_confidence",
    "gpt_target",
    "gpt_reason",
    "topic"
]


# =====================================================
# 1. 데이터 불러오기
# =====================================================

total_df = pd.read_excel(TOTAL_FILE)
labeled_df = pd.read_excel(LABELED_FILE)

print("통합 원본:", total_df.shape)
print("재분석 완료:", labeled_df.shape)


# =====================================================
# 2. 감정 라벨 정리
# =====================================================

def normalize_sentiment(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().lower()

    replace_map = {
        "긍정": "positive",
        "중립": "neutral",
        "부정": "negative"
    }

    return replace_map.get(x, x)


total_df["gpt_sentiment"] = total_df["gpt_sentiment"].apply(normalize_sentiment)
labeled_df["gpt_sentiment"] = labeled_df["gpt_sentiment"].apply(normalize_sentiment)


# =====================================================
# 3. 덮어씌울 대상 찾기
#    - 기존 통합 데이터에서 gpt_sentiment가 None 또는 neutral인 행
# =====================================================

text_valid_mask = (
    total_df["text_clean"].notna() &
    (total_df["text_clean"].astype(str).str.len() >= 2)
)

reanalyze_mask = (
    total_df["gpt_sentiment"].isna() |
    total_df["gpt_sentiment"].eq("neutral")
) & text_valid_mask

target_indices = total_df.index[reanalyze_mask].tolist()

print("덮어씌울 대상 수:", len(target_indices))
print("재분석 완료 데이터 수:", len(labeled_df))

if len(target_indices) != len(labeled_df):
    raise ValueError(
        f"행 개수가 맞지 않습니다. "
        f"덮어씌울 대상 {len(target_indices)}개, 재분석 완료 {len(labeled_df)}개"
    )


# =====================================================
# 4. 기존 gpt 컬럼만 덮어쓰기
# =====================================================

overwrite_cols = [
    "gpt_sentiment",
    "gpt_confidence",
    "gpt_target",
    "gpt_reason"
]

for col in overwrite_cols:
    total_df.loc[target_indices, col] = labeled_df[col].values


# =====================================================
# 5. 최종 컬럼만 남기기
# =====================================================

final_df = total_df[FINAL_COLUMNS].copy()

print("최종 데이터 shape:", final_df.shape)
print("최종 컬럼:")
print(final_df.columns.tolist())

print("\n최종 감정분포:")
print(final_df["gpt_sentiment"].value_counts(dropna=False))


# =====================================================
# 6. 저장
# =====================================================

# Excel 저장 오류 방지용 날짜 변환
for col in ["content_published_at", "comment_published_at"]:
    final_df[col] = pd.to_datetime(
        final_df[col],
        errors="coerce",
        utc=True
    ).dt.tz_convert(None)

final_df.to_csv(
    OUTPUT_FINAL_CSV,
    index=False,
    encoding="utf-8-sig"
)

final_df.to_excel(
    OUTPUT_FINAL_XLSX,
    index=False
)

print("✅ 최종 통합 데이터 저장 완료")
print("CSV:", OUTPUT_FINAL_CSV)
print("Excel:", OUTPUT_FINAL_XLSX)